In [1]:
import json, yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tbparse
from IPython.display import display

In [2]:
# Parameters
working_dir = "."
output_dir = "_output/run.cpu_TODAY"
results_dir = "_output/run.cpu_TODAY/result_csvs_and_pdfs"
summary_csv_path = "_output/run.cpu_TODAY/result_csvs_and_pdfs/summary.csv"
config_path = "_output/run.cpu_TODAY/config.yaml"
data_nicknames_path = "_output/run.cpu_TODAY/data_nicknames.json"

In [3]:
# load config file
with open(config_path, "r") as file:
    config_yaml = yaml.safe_load(file)
config_yaml_str = yaml.dump(config_yaml, default_flow_style=False, sort_keys=False)
print(config_yaml_str)

models:
- TraverseNN
- TraverseAvgPooling
train_data:
- 5leaf_100trees_pp_spr
test_data:
- 5leaf_100trees_pp_test_spr
device: cpu
timestamp: TODAY
cross_datasets: true
output_dir: _output
data_nicknames_path: data_nicknames.json
hyperparameter_optimize: true
n_hyperparameter_trials: 5
hyperparameters:
  learning_rate:
  - 0.0002
  - 0.0004
  batch_size:
  - 512
  accum_grad_batches:
  - 1
  epochs:
  - 100
  - 200
  feature_length:
  - 128
  dim_mlp_layers:
  - 256
working_dir: /home/lcollien/git/dpvt-experiments-1/train
start_time: 2025-01-31_11:12:22
output_path: _output
train_data_names:
- FourLeafTrain
test_data_names:
- FourLeafTest
use_cross_datasets: false
use_hyperparameter_optimize: true



In [4]:
# load data nickname paths
with open(data_nicknames_path, "r") as file:
    data_nicknames_json = json.load(file)
# data_nicknames_df = pd.DataFrame(data_nicknames_json)
display(data_nicknames_json)

{'data_dir': '../data',
 'FourLeafFourSiteTrain': '4leaf4site_train.p',
 'FourLeafFourSiteTest': '4leaf4site_test.p',
 'FourLeafTrain': '4leaf_train.p',
 'FourLeafTest': '4leaf_test.p',
 'TenLeafTrain': '10leaf_perfect_distinct_trees_train.p',
 'TenLeafTest': '10leaf_perfect_distinct_trees_test.p',
 'ThirtyLeafTest': '30leaf_perfect_distinct_trees_test.p',
 'ThirtyLeafTrain': '30leaf_perfect_distinct_trees_train.p',
 'harrington_subset_train': 'larch_harrington-small_2024-06-10_train.p',
 'harrington_subset_test': 'larch_harrington-small_2024-06-10_test.p',
 'Alisim10leaf_100sites_50algnmnts_test': 'larch_alisim_10_seq_100_sites_50_algnmnts_2024-11-13_test.p',
 'Alisim10leaf_100sites_50algnmnts_train': 'larch_alisim_10_seq_100_sites_50_algnmnts_2024-11-13_train.p',
 '10leaf_100trees_pp': '10leaf_perfect_100_distinct_trees.p',
 '10leaf_20trees_pp': '10leaf_perfect_20_distinct_trees.p',
 '10leaf_500trees_pp': '10leaf_perfect_500_distinct_trees.p',
 '10leaf_1000trees_pp': '10leaf_perfect_

In [5]:
# load results summary
summary_df = pd.read_csv(summary_csv_path)
display(summary_df)

,Unnamed: 0,model,train_data,test_data,device,timestamp,param_id,n_hyperparam_trials,learning_rate,batch_size,...,hyper_json_path,hyper_llog_path,hyper_benchmark_path,hyper_checkpoint_path,train_llog_path,train_benchmark_path,train_clog_path,train_checkpoint_path,test_llog_path,test_checkpoint_path
0,0,TraverseNN,alisim_10_seq_20_sites_200_alignments_train,alisim_10_seq_20_sites_200_alignments_test,cpu,TODAY,ParamOpt,5,0.005922,16,...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...


In [6]:
# Add summary columns
summary_df['label'] = summary_df[['model','train_data','test_data', 'param_id']].apply(lambda x:'\n'.join(x), axis=1)
summary_df['percent_epochs'] = summary_df['train_epochs']/summary_df['max_epochs']

# Define the custom function to extract num_leaves
def extract_num_leaves(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim and perfect phylogeny data that follows specific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'alisim' in row:
        return row.split('_')[1]  # Take the second entry after splitting by '_'
    elif 'leaf' in row:
        return row.split('leaf')[0]  # Take the part after 'leaf'
    else:
        return None  # Return None if neither condition is met

def extract_num_sites(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim data that follows sepcific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'alisim' in row:
        return row.split("_")[3]
    else:
        return None


def extract_num_trees(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim data that follows sepcific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'pp' in row:
        return row.split("trees")[0].split("_")[-1]
    else:
        return None


summary_df["train_num_leaves"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_leaves), errors="coerce")
summary_df["train_num_sites"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_sites), errors="coerce")
summary_df["train_num_trees"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_trees), errors="coerce")
summary_df["test_num_leaves"] = pd.to_numeric(summary_df["test_data"].apply(extract_num_leaves), errors="coerce")
summary_df["model_and_train_data"] = summary_df['model'].astype(str) + "-" + summary_df['train_data'].astype(str)

# Sort models
model_order = ["TraverseMaxPooling", "TraverseAvgPooling", "TraverseNN", "TransformerEncoderTraversal"]
summary_df['model'] = pd.Categorical(summary_df['model'], categories=model_order, ordered=True)

num_runs = len(summary_df)
summary_df

,Unnamed: 0,model,train_data,test_data,device,timestamp,param_id,n_hyperparam_trials,learning_rate,batch_size,...,train_checkpoint_path,test_llog_path,test_checkpoint_path,label,percent_epochs,train_num_leaves,train_num_sites,train_num_trees,test_num_leaves,model_and_train_data
0,0,TraverseNN,alisim_10_seq_20_sites_200_alignments_train,alisim_10_seq_20_sites_200_alignments_test,cpu,TODAY,ParamOpt,5,0.005922,16,...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,/home/lcollien/git/dpvt-experiments-1/train/_o...,TraverseNN\nalisim_10_seq_20_sites_200_alignme...,1.0,10,20,NaN,10,TraverseNN-alisim_10_seq_20_sites_200_alignmen...


In [7]:
# get dataframe from lightning log

def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [25]:
# get dfs from logs

hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}
custom_dfs = {}

for index,row in summary_df.iterrows():
    label_str = row.label
    label = (row.model,row.train_data,row.test_data,row.param_id)
    train_df = get_df_from_log(row.train_llog_path)
    train_dfs[label] = train_df
    test_df = get_df_from_log(row.test_llog_path)
    test_dfs[label] = test_df
    custom_df = get_df_from_log(row.train_clog_path)
    custom_dfs[label] = custom_df
    hyperparam_dfs[label] = {}
    for i in range(row.n_hyperparam_trials):
        print(row.hyper_llog_path)
        hyperparam_dfs[label][i] = get_df_from_log(f'{row.hyper_llog_path}/version_{i}')


/home/lcollien/git/dpvt-experiments-1/train/_output/run.cpu_TODAY/lightning_logs/optimize_hyperparameters/TraverseNN-alisim_10_seq_20_sites_200_alignments_train-ParamOpt
/home/lcollien/git/dpvt-experiments-1/train/_output/run.cpu_TODAY/lightning_logs/optimize_hyperparameters/TraverseNN-alisim_10_seq_20_sites_200_alignments_train-ParamOpt
/home/lcollien/git/dpvt-experiments-1/train/_output/run.cpu_TODAY/lightning_logs/optimize_hyperparameters/TraverseNN-alisim_10_seq_20_sites_200_alignments_train-ParamOpt
/home/lcollien/git/dpvt-experiments-1/train/_output/run.cpu_TODAY/lightning_logs/optimize_hyperparameters/TraverseNN-alisim_10_seq_20_sites_200_alignments_train-ParamOpt
/home/lcollien/git/dpvt-experiments-1/train/_output/run.cpu_TODAY/lightning_logs/optimize_hyperparameters/TraverseNN-alisim_10_seq_20_sites_200_alignments_train-ParamOpt


In [9]:
# reshape data

models = ['model', 'train_data', 'test_data', 'param_id']
hyperparams = ['learning_rate', 'batch_size', 'accum_grad_batches', 'max_epochs', 'feature_length', 'dim_mlp_layers']

# only keep one row per train_data

# train_df = summary_df
# train_df['label'] = train_df['model'].astype(str) + "-" + train_df['train_data'].astype(str)
train_df = summary_df.groupby('model_and_train_data')[hyperparams].first().reset_index()

melt_df = pd.melt(train_df, id_vars='model_and_train_data', var_name='Metric', value_name='Value')
melt_df = melt_df[melt_df.Metric.isin(hyperparams)]
display(melt_df)

,model_and_train_data,Metric,Value
0,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,learning_rate,0.005922
1,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,batch_size,16.000000
2,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,accum_grad_batches,1.000000
3,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,max_epochs,28.000000
4,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,feature_length,4.000000
5,TraverseNN-alisim_10_seq_20_sites_200_alignmen...,dim_mlp_layers,256.000000


# Testing performance (AUROC and test loss)

In [10]:
# heatmap of AUROCs
for value in ['test_auroc', 'test_loss']:
    df_sorted = summary_df.sort_values(by=['model', 'train_num_leaves', 'train_num_sites'])
    plt.figure(figsize=(10,8))
    indices = ['model']
    extra_cols = ['train_num_leaves', 'train_num_trees', 'train_num_sites']
    for col_name in extra_cols:
        if not df_sorted[col_name].isnull().values.any():
            indices.append(col_name)
    if len(indices) > 1:
        heatmap_data = df_sorted.pivot_table(
            index=indices,
            columns=['test_num_leaves'],
            values=value
        )
    else:
        heatmap_data = df_sorted.pivot(index='model', columns="test_data", values=value)
    ax = sns.heatmap(
        heatmap_data,
        annot=True,
        cbar_kws={'label': f'{value}'},
    )
    ax.set_xlabel("Number of leaves in test data (200 alignments)")
    ax.set_ylabel("Model - number of leaves - number of trees")
    plt.tight_layout()
    plt.savefig(f"{results_dir}/{value}_heatmap.pdf")
    plt.show()

/tmp/ipykernel_22993/2800339514.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  heatmap_data = df_sorted.pivot_table(
/tmp/ipykernel_22993/2800339514.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  heatmap_data = df_sorted.pivot_table(


In [11]:
# barplot hyperparameters combined

df = melt_df
print(df.columns)
if len(df['model_and_train_data'].unique()) < 10:
    num_params = len(hyperparams)
    plt.figure(figsize=(3 * num_params,8))
    sns.barplot(x='Metric', y='Value', hue='model_and_train_data', data=df, palette='deep')
    # plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.4), ncol=num_runs/2)
    plt.xticks(rotation=45)
    plt.title('Hyper Parameters by Model')
    plt.tight_layout()
    plt.savefig(f'{results_dir}/hyperparameter_summary.barplot.pdf')
    plt.show()
    plt.clf()
else:
    print("To many trained models, don't plot hyperparameter summary")

Index(['model_and_train_data', 'Metric', 'Value'], dtype='object')


## Training and Hyperparameter settings and results

In [12]:
# individual bar plots
x_measures = {
  "Training Runtime (in secs)": "train_walltime",
  "Learning Rate": "learning_rate",
  "Batch Size": "batch_size",
  "Gradient Accumulation": "accum_grad_batches",
  "Max Epochs": "max_epochs",
  "Train Epochs": "train_epochs",
  "Train Steps": "train_steps",
  "Percent of Max Epochs Used": "percent_epochs",
}

model_list = summary_df['model'].unique()

for x_title,x_col in x_measures.items():
  fig, axes = plt.subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
  x_min = summary_df[x_measures[x_title]].min() * 0.99
  x_max = summary_df[x_measures[x_title]].max() * 1.01
  if not isinstance(axes, list):
    axes = [axes]
  for ax, model in zip(axes, model_list):
      this_df = summary_df[summary_df['model']==model]
      sns.barplot(y="train_data", x=f"{x_col}", data=this_df, palette='deep', ax=ax)
      ax.set_xlabel(f"{x_title}")
      ax.set_title(f"{model}")
      ax.set_xlim(x_min, x_max)
      plt.tight_layout()
  plt.savefig(f'{results_dir}/{x_col}.barplot.pdf')
  plt.show()
  plt.clf()


/home/lcollien/.local/lib/python3.9/site-packages/seaborn/categorical.py:645: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  g_vals = grouped_vals.get_group(g)
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/categorical.py:645: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  g_vals = grouped_vals.get_group(g)
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/categorical.py:645: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  g_vals = grouped_vals.get_group(g)
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/catego

# More runtime plots

In [13]:
# line plots over epochs (using custom logs)

measures = {
  "Runtime over Batches": "walltime_per_batch",
  "Loss over Batches": "loss_per_batch",
  "Runtime over Epochs": "walltime_per_epoch",
  "Loss over Epochs": "avgloss_per_epoch",
}

df_list = []
for key,df in custom_dfs.items():
  this_df = df
  df['model_and_train_data'] = key[0] + "-" + key[1]
  df['model'] = key[0]
  df_list.append(this_df)

runtime_df = pd.concat(df_list)

for title,col in measures.items():
  fig, axes = plt.subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
  this_tag_df = runtime_df[runtime_df.tag==col]
  x_min = 0
  x_max = this_tag_df["step"].max() + 2
  if not isinstance(axes, list):
    axes = [axes]
  for ax, model in zip(axes, model_list):
      this_df = this_tag_df[this_tag_df['model']==model]
      plot = sns.lineplot(y="value", x="step", data=this_df, hue="model_and_train_data", palette='deep', ax=ax, legend=True)
      ax.set_xlabel("Training Step")
      ax.set_ylabel(f"{title}")
      ax.set_title(f"{model}")
      ax.set_xlim(x_min, x_max)
      plt.tight_layout()
      handles, labels = plot.get_legend_handles_labels()  # Get legend info from subplot
      ax.get_legend().remove()
  fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))

  plt.savefig(f'{results_dir}/{col}.barplot.pdf')
  plt.show()
  plt.clf()

/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/home/lcollien/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will

## Loss over walltime

In [14]:
# line plot loss over walltime (using custom logs)
fig, axes = plt.subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
if not isinstance(axes, list):
    axes = [axes]
model_list = list(model_list)
label_set = set()
for key,df in custom_dfs.items():
    label = key[0] + "-" + key[1]
    if label in label_set:
        continue
    label_set.add(label)
    df_time = df[df.tag == 'walltime_per_batch']
    df_loss = df[df.tag == 'loss_per_batch']
    ax = axes[model_list.index(key[0])]
    ax.plot(df_time.value, df_loss.value, label=key[1])
    ax.set_title(key[0])

fig.supylabel("Training loss", fontsize=12)
fig.supxlabel('Walltime (in seconds)', fontsize=12)

x_title = "Training Loss over Time"
plt.xticks(rotation=45)

fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig(f'{results_dir}/loss_per_walltime.lineplot.pdf')
plt.show()
plt.clf()


## Training loss, positive, and negative predictions over epochs

In [15]:
# line plots over epochs (using lightning logs)

x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    fig, axes = plt.subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
    if not isinstance(axes, list):
        axes = [axes]
    label_set = set()
    for key,df in train_dfs.items():
        label = key[0] + "-" + key[1]
        if label in label_set:
            continue
        label_set.add(label)
        ax = axes[model_list.index(key[0])]
        this_df = df[df.tag == x_col]
        ax.plot(this_df.step, this_df.value, label=key[1])
        ax.set_title(key[0])

    fig.supylabel(f"{x_title}", fontsize=12)
    fig.supxlabel('Epochs', fontsize=12)
    fig.suptitle(f"{x_title} by Model")

    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.lineplot.pdf')
    plt.show()
    plt.clf()

## Training loss over epochs in hyperparameter trials

In [23]:
# line plot hyperparam training

x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

label_set = set()
for x_title,x_col in x_measures.items():
    for label,hp_dict in hyperparam_dfs.items():
      model_and_train_label = label[0] + "-" + label[1]
      if model_and_train_label in label_set:
        continue
      label_set.add(model_and_train_label)
      label_str = ','.join(label)
      print(hp_dict)
      for key,df in hp_dict.items():
          df_tag = df[df.tag == x_col]
          plt.plot(df_tag.step, df_tag.value, label=key)

      plt.xlabel("Epochs")
      plt.ylabel(f"{x_title}")
      plt.title(f"{x_title} by Model\n{model_and_train_label}")
      plt.xticks(rotation=45)

      plt.legend(loc="upper right")
      plt.tight_layout()
      plt.savefig(f'{results_dir}/hyperparameter_opt.{x_col}.lineplot.pdf')
      plt.show()
      plt.clf()

{0: Empty DataFrame
Columns: []
Index: [], 1:     step               tag     value
0      1             epoch  0.000000
1      1             epoch  0.000000
2      3             epoch  1.000000
3      3             epoch  1.000000
4      5             epoch  2.000000
5      5             epoch  2.000000
6      7             epoch  3.000000
7      7             epoch  3.000000
8      9             epoch  4.000000
9      9             epoch  4.000000
10    11             epoch  5.000000
11    11             epoch  5.000000
12    13             epoch  6.000000
13    13             epoch  6.000000
14    15             epoch  7.000000
15    15             epoch  7.000000
16     0              loss  0.278025
17     0              loss  0.288378
18     0              loss  0.277033
19     1              loss  0.268078
20     1              loss  0.266624
21     2              loss  0.259043
22     2              loss  0.257979
23     3              loss  0.251941
24     3              loss  0

AttributeError: 'DataFrame' object has no attribute 'tag'